# MMLM: train the transformer expert on a Colab GPU

MMLM-CM mixes counting experts with one neural network. The network trained so far is a small LSTM (806,144 weights, 40 minutes on two CPU cores). This notebook trains a larger one, a byte-level transformer, on a GPU, and measures the mixture with it on the development half of the validation text.

**Before running:** Runtime → Change runtime type → GPU. The test split is never read here.

When it finishes you get two files: the weights (`.pt`) and a record (`.json`). Bring both back to the repository (`benchmarks/mmlm/`) so the result can be tested once on the test split under a preregistration.


In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: change the runtime type')


In [ ]:
!git clone --depth 1 --branch stage https://github.com/psalmprax/science.git
%cd science
!pip -q install datasets


## Train

`--hours` is the wall-clock budget; the weights that score best on held-out text are kept, so stopping early loses nothing but time. On a free T4, 2 hours with the defaults (8 layers, width 512, about 25 million weights) reads roughly a billion bytes.

`--corpus wikitext103` trains on 540 MB (fifty times the text the CPU network saw). `--corpus wikitext2` is the quick check: 10.8 MB, where a network this size overfits within minutes.


In [ ]:
!python scripts/train_mmlm_gpu_expert.py --hours 2 --corpus wikitext103


## What the numbers mean

All in bits per byte on the development half (lower is better). The references printed at the end are the same quantities for the CPU model: the mixture with the LSTM (1.4757), the LSTM alone (1.7890), and the best classical compressor given the same training text (1.5363).


In [ ]:
import json
record = json.load(open('benchmarks/mmlm/transformer_expert_wikitext103.json'))
print(json.dumps({k: record[k] for k in ('parameters', 'steps', 'bytes_seen', 'seconds', 'gpu', 'best_valid_bits_per_byte', 'development')}, indent=2))


## Take the files with you


In [ ]:
from google.colab import files
files.download('benchmarks/mmlm/transformer_expert_wikitext103.pt')
files.download('benchmarks/mmlm/transformer_expert_wikitext103.json')
